In [24]:
import pandas as pd
import numpy as np

#1. تحميل بيانات التدريب
df_train = pd.read_csv('train-test.csv')

# 2. عرض أول 5 أسطر من البيانات
print("--- معاينة أول 5 أسطر ---")
display(df_train.head())

# 3. عرض معلومات الأعمدة والقيم المفقودة
print("\n--- معلومات الأعمدة والأنواع ---")
df_train.info()

# 4. ملخص إحصائي للقيم العددية
print("\n--- ملخص إحصائي ---")
display(df_train.describe())

--- معاينة أول 5 أسطر ---


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28



--- معلومات الأعمدة والأنواع ---
<class 'pandas.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  str    
 1   pickup        48000 non-null  str    
 2   delivery      48000 non-null  str    
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  str    
 9   weight        47700 non-null  float64
 10  date          48000 non-null  str    
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), str(5)
memory usage: 7.1 MB

--- ملخص إحصائي ---


,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,posted_rate
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,47700.000000,47626.000000,48000.000000,48000.000000
mean,35.647545,-90.928964,35.641175,-90.857310,1135.856654,31028.844004,1.083387,2.062468,2373.980682
std,4.315285,13.482431,4.317199,13.476589,728.564416,9391.440620,0.168091,0.291391,1486.493245
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,0.949670,1.891030,1251.555000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.500000,1.055800,2.055750,2030.760000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.525000,37018.000000,1.219590,2.221685,3330.750000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,1.467780,3.610350,25533.000000


In [25]:
# 1. معالجة القيم الشاذة والمفقودة في بيانات التدريب
# تحويل الوزن السلبي إلى قيمة مطلقة أو تعبئة المفقود بالوسيط (Median)
df_train['weight'] = df_train['weight'].abs()
df_train['weight'] = df_train['weight'].fillna(df_train['weight'].median())
df_train['market_index'] = df_train['market_index'].fillna(df_train['market_index'].median())

# 2. تحويل التواريخ واستخراج الميزات (Feature Engineering)
if 'pickup_date' in df_train.columns:
    df_train['pickup_date'] = pd.to_datetime(df_train['pickup_date'])
    df_train['month'] = df_train['pickup_date'].dt.month
    df_train['day'] = df_train['pickup_date'].dt.day
    df_train['dayofweek'] = df_train['pickup_date'].dt.dayofweek

print("--- تم تنظيف البيانات واستخراج الميزات بنجاح ---")
print("عدد القيم المفقودة بعد التنظيف:")
print(df_train.isnull().sum())

--- تم تنظيف البيانات واستخراج الميزات بنجاح ---
عدد القيم المفقودة بعد التنظيف:
load_id         0
pickup          0
delivery        0
pickup_lat      0
pickup_lon      0
delivery_lat    0
delivery_lon    0
distance        0
equipment       0
weight          0
date            0
market_index    0
quote_signal    0
posted_rate     0
dtype: int64


In [26]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# 1. معالجة التاريخ لبيانات التدريب
df_train['date'] = pd.to_datetime(df_train['date'])
df_train['month'] = df_train['date'].dt.month
df_train['day'] = df_train['date'].dt.day
df_train['dayofweek'] = df_train['date'].dt.dayofweek

# 2. ترميز الأعمدة النصية (Categorical Encoding)
cat_cols = ['pickup', 'delivery', 'equipment']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    # نحول القيم إلى نصوص لتفادي أي أخطاء
    df_train[col] = le.fit_transform(df_train[col].astype(str))
    label_encoders[col] = le

# تحديد الميزات (Features) والهدف (Target)
features = ['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 
            'distance', 'equipment', 'weight', 'market_index', 'quote_signal', 
            'month', 'day', 'dayofweek', 'pickup', 'delivery']

X = df_train[features]
y = df_train['posted_rate']

print("جاري تدريب نموذج Random Forest...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X, y)
print("--- تم تدريب النموذج بنجاح! ---")

# 3. تحميل وتجهيز بيانات Validation وتوليد التوقعات
df_val = pd.read_csv('validation.csv')

# تنظيف وتجهيز Validation
df_val['weight'] = df_val['weight'].abs().fillna(df_train['weight'].median())
df_val['market_index'] = df_val['market_index'].fillna(df_train['market_index'].median())

df_val['date'] = pd.to_datetime(df_val['date'])
df_val['month'] = df_val['date'].dt.month
df_val['day'] = df_val['date'].dt.day
df_val['dayofweek'] = df_val['date'].dt.dayofweek

for col in cat_cols:
    le = label_encoders[col]
    # التعامل مع القيم الجديدة غير الموجودة سابقاً
    df_val[col] = df_val[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

# التنبؤ على بيانات Validation
X_val = df_val[features]
df_val['predicted_rate'] = model.predict(X_val)

# إنشاء ملف التوقعات النهائي المطلوب بالشروط الدقيقة
val_predictions = df_val[['load_id', 'predicted_rate']]
val_predictions.to_csv('validation_predictions.csv', index=False)

print("\n تم إنشاء ملف validation_predictions.csv بنجاح!")
print("عدد أسطر الملف الناتج:", len(val_predictions))
display(val_predictions.head())

جاري تدريب نموذج Random Forest...
--- تم تدريب النموذج بنجاح! ---

 تم إنشاء ملف validation_predictions.csv بنجاح!
عدد أسطر الملف الناتج: 12000


,load_id,predicted_rate
0,TE-000001,813.4148
1,TE-000002,4917.7984
2,TE-000003,5097.5592
3,TE-000004,4232.6516
4,TE-000005,1822.6835


In [27]:
# 1. تحميل وتجهيز بيانات ديسمبر
df_dec = pd.read_csv('december-chart-inputs.csv')

# تنظيف وتنسيق التواريخ لبيانات ديسمبر
df_dec['weight'] = df_dec['weight'].abs().fillna(df_train['weight'].median())
df_dec['market_index'] = df_dec['market_index'].fillna(df_train['market_index'].median())

df_dec['date'] = pd.to_datetime(df_dec['date'])
df_dec['month'] = df_dec['date'].dt.month
df_dec['day'] = df_dec['date'].dt.day
df_dec['dayofweek'] = df_dec['date'].dt.dayofweek

for col in cat_cols:
    le = label_encoders[col]
    df_dec[col] = df_dec[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

# التنبؤ على بيانات ديسمبر
X_dec = df_dec[features]
df_dec['predicted_rate'] = model.predict(X_dec)

# حفظ ملف توقعات ديسمبر
dec_predictions = df_dec[['load_id', 'predicted_rate']]
dec_predictions.to_csv('december_predictions.csv', index=False)
print("تم حفظ ملف december_predictions.csv بنجاح!")

# 2. تشغيل سكربت التقييم score.py لإنشاء الرسم البياني
import subprocess
result = subprocess.run([
    'python', 'score.py', 
    '--predictions', 'validation_predictions.csv', 
    '--december-predictions', 'december_predictions.csv'
], capture_output=True, text=True)

print("\n--- مخرجات سكربت التقييم ---")
print(result.stdout)
if result.stderr:
    print("ملاحظات/أخطاء:", result.stderr)

KeyError: 'market_index'

In [28]:
import subprocess

# 1. تحميل وتجهيز بيانات ديسمبر
df_dec = pd.read_csv('december-chart-inputs.csv')

# معالجة الوزن إذا كان موجوداً
if 'weight' in df_dec.columns:
    df_dec['weight'] = df_dec['weight'].abs().fillna(df_train['weight'].median())

# معالجة التواريخ
if 'date' in df_dec.columns:
    df_dec['date'] = pd.to_datetime(df_dec['date'])
    df_dec['month'] = df_dec['date'].dt.month
    df_dec['day'] = df_dec['date'].dt.day
    df_dec['dayofweek'] = df_dec['date'].dt.dayofweek

# ترميز الأعمدة النصية المتاحة
for col in cat_cols:
    if col in df_dec.columns:
        le = label_encoders[col]
        df_dec[col] = df_dec[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

# مطابقة الميزات المتاحة فقط في ملف ديسمبر
available_features = [f for f in features if f in df_dec.columns]

# إذا كانت بعض الميزات مفقودة في ملف ديسمبر، نضع لها قيماً افتراضية لتوافق النموذج
for f in features:
    if f not in df_dec.columns:
        df_dec[f] = df_train[f].median() if df_train[f].dtype in ['int64', 'float64'] else -1

# التنبؤ على بيانات ديسمبر
X_dec = df_dec[features]
df_dec['predicted_rate'] = model.predict(X_dec)

# حفظ ملف توقعات ديسمبر
dec_predictions = df_dec[['load_id', 'predicted_rate']]
dec_predictions.to_csv('december_predictions.csv', index=False)
print("--- تم حفظ ملف december_predictions.csv بنجاح! ---")

# 2. تشغيل سكربت التقييم score.py لإنشاء الرسم البياني
result = subprocess.run([
    'python', 'score.py', 
    '--predictions', 'validation_predictions.csv', 
    '--december-predictions', 'december_predictions.csv'
], capture_output=True, text=True)

print("\n--- مخرجات سكربت التقييم الرسمي ---")
print(result.stdout)
if result.stderr:
    print("ملاحظات:", result.stderr)
    

KeyError: "['load_id'] not in index"

In [ ]:
import subprocess

# 1. تحميل وتجهيز بيانات ديسمبر
df_dec = pd.read_csv('december-chart-inputs.csv')

# معالجة الوزن إن وجد
if 'weight' in df_dec.columns:
    df_dec['weight'] = df_dec['weight'].abs().fillna(df_train['weight'].median())

# معالجة التواريخ إن وجدت
if 'date' in df_dec.columns:
    df_dec['date'] = pd.to_datetime(df_dec['date'])
    df_dec['month'] = df_dec['date'].dt.month
    df_dec['day'] = df_dec['date'].dt.day
    df_dec['dayofweek'] = df_dec['date'].dt.dayofweek

# ترميز الأعمدة النصية المتاحة
for col in cat_cols:
    if col in df_dec.columns:
        le = label_encoders[col]
        df_dec[col] = df_dec[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

# إضافة الميزات المفقودة إن وجدت بقيم افتراضية
for f in features:
    if f not in df_dec.columns:
        df_dec[f] = df_train[f].median() if df_train[f].dtype in ['int64', 'float64'] else -1

# التنبؤ على بيانات ديسمبر
X_dec = df_dec[features]
df_dec['predicted_rate'] = model.predict(X_dec)

# التأكد من وجود عمود load_id أو إنشائه تلقائياً
if 'load_id' not in df_dec.columns:
    df_dec['load_id'] = [f"DEC-{i+1:06d}" for i in range(len(df_dec))]

# حفظ ملف توقعات ديسمبر
dec_predictions = df_dec[['load_id', 'predicted_rate']]
dec_predictions.to_csv('december_predictions.csv', index=False)
print("--- تم حفظ ملف december_predictions.csv بنجاح! ---")

# 2. تشغيل سكربت التقييم score.py إذا كان الملف موجوداً
import os
if os.path.exists('score.py'):
    result = subprocess.run([
        'python', 'score.py', 
        '--predictions', 'validation_predictions.csv', 
        '--december-predictions', 'december_predictions.csv'
    ], capture_output=True, text=True)

    print("\n--- مخرجات سكربت التقييم الرسمي ---")
    print(result.stdout)
    if result.stderr:
        print("ملاحظات:", result.stderr)
else:
    print("\nتنفيذي: تم حفظ ملفات التوقعات بنجاح. يرجى تنزيل ملف score.py وإضافته للمجلد لتشغيل التقييم الرسمي.")